## tl;dr

Amb un missatge avançat tipus de 8.000 tokens d'entrada i 1.200 de sortida, Haiku 4.5 costa aproximadament 0,014 USD; Groq Qwen3 32B, 0,003028 USD; i Cloudflare Qwen3 30B A3B, 0,00081 USD. Una GPU de 16 GB a 0,58 USD/h només compensa en cost pur si manté una utilització molt alta; per al pilot convé pagar per ús i conservar fallback.

## Context & Methods

Lector: fundador i equip de producte de Noesis. Decisió: com oferir IA útil des del primer dia sense destruir marge ni fiabilitat. Les tarifes són les publicades pels proveïdors el 14-07-2026 i s'han de revisar abans de contractar.

### Key Assumptions

- Un missatge avançat agrega totes les rondes d'eines: 8.000 tokens d'entrada i 1.200 de sortida.
- Els límits mensuals actuals són 75, 300 i 1.500 missatges avançats.
- No s'inclou IVA, canvi EUR/USD, embeddings, àudio, suport ni enginyeria.
- El cost no demostra qualitat: els models s'han de validar amb ordres reals en castellà i català.

## Data

Fonts oficials: Anthropic Haiku, Groq pricing, Cloudflare Workers AI pricing i Runpod Serverless. URLs i data de revisió queden en els paràmetres executables.

In [1]:
reviewed_on = '2026-07-14'
sources = {
    'anthropic': 'https://www.anthropic.com/claude/haiku',
    'groq': 'https://groq.com/pricing',
    'cloudflare': 'https://developers.cloudflare.com/workers-ai/platform/pricing/',
    'runpod': 'https://www.runpod.io/product/serverless',
}
providers = {
    'Anthropic Haiku 4.5': {'input': 1.00, 'output': 5.00},
    'Groq Qwen3 32B': {'input': 0.29, 'output': 0.59},
    'Groq Llama 3.3 70B': {'input': 0.59, 'output': 0.79},
    'Cloudflare Qwen3 30B A3B': {'input': 0.051, 'output': 0.335},
}
input_tokens = 8_000
output_tokens = 1_200
plan_credits = {'Autónomo': 75, 'Negocio': 300, 'Sin Límites': 1_500}
reviewed_on, sources

('2026-07-14',
 {'anthropic': 'https://www.anthropic.com/claude/haiku',
  'groq': 'https://groq.com/pricing',
  'cloudflare': 'https://developers.cloudflare.com/workers-ai/platform/pricing/',
  'runpod': 'https://www.runpod.io/product/serverless'})

## Results

In [2]:
def message_cost(rate):
    return (input_tokens * rate['input'] + output_tokens * rate['output']) / 1_000_000

rows = []
for provider, rate in providers.items():
    per_message = message_cost(rate)
    row = {'provider': provider, 'cost_per_advanced_message_usd': round(per_message, 6)}
    for plan, credits in plan_credits.items():
        row[f'{plan}_monthly_cap_usd'] = round(per_message * credits, 3)
    rows.append(row)
rows

[{'provider': 'Anthropic Haiku 4.5',
  'cost_per_advanced_message_usd': 0.014,
  'Autónomo_monthly_cap_usd': 1.05,
  'Negocio_monthly_cap_usd': 4.2,
  'Sin Límites_monthly_cap_usd': 21.0},
 {'provider': 'Groq Qwen3 32B',
  'cost_per_advanced_message_usd': 0.003028,
  'Autónomo_monthly_cap_usd': 0.227,
  'Negocio_monthly_cap_usd': 0.908,
  'Sin Límites_monthly_cap_usd': 4.542},
 {'provider': 'Groq Llama 3.3 70B',
  'cost_per_advanced_message_usd': 0.005668,
  'Autónomo_monthly_cap_usd': 0.425,
  'Negocio_monthly_cap_usd': 1.7,
  'Sin Límites_monthly_cap_usd': 8.502},
 {'provider': 'Cloudflare Qwen3 30B A3B',
  'cost_per_advanced_message_usd': 0.00081,
  'Autónomo_monthly_cap_usd': 0.061,
  'Negocio_monthly_cap_usd': 0.243,
  'Sin Límites_monthly_cap_usd': 1.215}]

In [3]:
gpu_hourly_usd = 0.58
always_on_monthly_usd = gpu_hourly_usd * 730
break_even = {
    provider: round(always_on_monthly_usd / message_cost(rate))
    for provider, rate in providers.items()
}
serverless_per_message = {
    f'{seconds}s GPU actiu': round(gpu_hourly_usd * seconds / 3600, 6)
    for seconds in (10, 30)
}
{'always_on_monthly_usd': round(always_on_monthly_usd, 2), 'break_even_messages': break_even, 'serverless': serverless_per_message}

{'always_on_monthly_usd': 423.4,
 'break_even_messages': {'Anthropic Haiku 4.5': 30243,
  'Groq Qwen3 32B': 139828,
  'Groq Llama 3.3 70B': 74700,
  'Cloudflare Qwen3 30B A3B': 522716},
 'serverless': {'10s GPU actiu': 0.001611, '30s GPU actiu': 0.004833}}

## Takeaways

1. El cost de tokens no és el principal risc del pilot: ho són qualitat d'eines, privacitat, latència i fallades.
2. Groq o Cloudflare poden abaratir el primer intent, però el tram gratuït no és un SLA ni una base comercial.
3. L'autoallotjament 24/7 no compensa amb poc volum; serverless pot competir si el model respon bé i els temps actius són curts.
4. Recomanació: regles locals → model privat opcional → model obert de pagament per ús → Haiku de fallback, sempre amb consentiment, límit i telemetria per negoci.